# Getting Started with Graflow — Hands-on Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GraflowAI/graflow-examples/blob/main/notebooks/hands_on_guide.ipynb)

An interactive tutorial that walks you through Graflow's core features on Google Colab.

## What You'll Learn

| # | Topic | Description |
|---|---|---|
| 1 | Your First Graph | Minimal workflow with `@task` + `>>` operator |
| 2 | Parallel Execution | Diamond pattern (Fan-out → Fan-in) with the `\|` operator |
| 3 | Data Sharing | Channel-based inter-task communication + concurrency-safe primitives + automatic keyword argument resolution |
| 4 | Branching & Loops | `next_task()` / `next_iteration()` / `max_cycles` / `RetryPolicy` |
| 5 | Parallel Error Policies | Best-effort / At-least-N / Critical policies |
| 6 | Hands-on Exercise | Data analysis pipeline (putting it all together) |
| 7 | Checkpoint/Resume | Creating and restoring checkpoints |
| 8 | Custom Task Handlers | Swappable execution strategies (e.g., TimingHandler) |
| 9 | LLM Integration | LLM workflows via `inject_llm_client` (Ollama + LiteLLM) + Google ADK / Pydantic AI agents |

> **Note**: The LLM integration section (Section 9) uses **Ollama** (local LLM), so you can run every cell without an API key.


## 0. Setup

In [ ]:
# Install system libraries required to build pygraphviz
!apt-get update -qq && apt-get install -y -qq --no-install-recommends graphviz graphviz-dev > /dev/null 2>&1

# Install optional dependencies for ADK, Pydantic AI, and visualization
# Full list of extras: https://graflow.ai/docs/getting-started/installation#install-with-all-extras
!pip install -q 'graflow[adk,pydantic-ai,visualization]>=0.1.8'


In [ ]:
# Verify installation
import graflow

print(f"Graflow version: {graflow.__version__}")

---

## 1. Your First Graph — No Upfront State Definition Required

Build and run a workflow in three steps:

1. Decorate functions with `@task`
2. Define dependencies with the `>>` operator
3. Run with `ctx.execute()`

**No upfront state schema (TypedDict) or `compile()` step needed.**

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("my_pipeline") as ctx:

    @task(inject_context=True)
    def task_a(context: TaskExecutionContext):
        channel = context.get_channel()
        text = channel.get("text", "")
        channel.set("text", text + "a")

    @task(inject_context=True)
    def task_b(context: TaskExecutionContext):
        channel = context.get_channel()
        text = channel.get("text", "")
        channel.set("text", text + "b")

    # Define execution order with >>
    task_a >> task_b

    # Execute (ret_context=True gives access to the channel)
    _, exec_ctx = ctx.execute("task_a", ret_context=True)
    print(exec_ctx.get_channel().get("text"))  # 'ab'

### Low-Level API: `add_node` / `add_edge` Style

Graflow also supports a low-level API via `add_node` / `add_edge` for dynamic graph construction and fine-grained control.

In [ ]:
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph
from graflow.core.task import TaskWrapper

# Build a graph in LangGraph-like style
graph = TaskGraph()

extract = TaskWrapper("extract", func=lambda: print("  extract"), register_to_context=False)
transform = TaskWrapper("transform", func=lambda: print("  transform"), register_to_context=False)
load = TaskWrapper("load", func=lambda: print("  load"), register_to_context=False)

graph.add_node(extract, "extract")
graph.add_node(transform, "transform")
graph.add_node(load, "load")

graph.add_edge("extract", "transform")
graph.add_edge("transform", "load")

# Execute
context = ExecutionContext.create(graph, "extract", max_steps=10)
engine = WorkflowEngine()
engine.execute(context)

---

## 2. Parallel Execution — Readable Structure in One Line

In Graflow, `>>` means sequential and `|` means parallel. **A Diamond (Fan-out → Fan-in) pattern fits in a single line:**

```
fetch >> (transform_a | transform_b) >> store
```

In [ ]:
from graflow import task, workflow

with workflow("diamond") as ctx:

    @task
    def fetch():
        print("  Fetching data")

    @task
    def transform_a():
        print("  Transform A")

    @task
    def transform_b():
        print("  Transform B")

    @task
    def store():
        print("  Storing results")

    # Diamond pattern in a single line
    fetch >> (transform_a | transform_b) >> store

    ctx.execute("fetch")

### Dynamic Task Lists with Functional Style

You can also build task lists dynamically using the `chain()` / `parallel()` functions.

In [ ]:
from graflow import parallel, task, workflow

with workflow("dynamic_parallel") as ctx:

    @task
    def start():
        print("  Started")

    # Generate tasks dynamically
    worker_tasks = []
    for i in range(4):

        @task(id=f"worker_{i}")
        def worker(task_id=i):
            print(f"  Worker {task_id} running")

        worker_tasks.append(worker)

    @task
    def aggregate():
        print("  Aggregating results")

    # Build dynamically with chain + parallel
    start >> parallel(*worker_tasks) >> aggregate

    ctx.execute("start")

---

## 3. Inter-Task Data Sharing — No Reducer Needed

Graflow uses a **Channel (key-value store)** for explicit reads and writes between tasks. Since each task controls what it updates and when, implicit merge rules (reducers) are unnecessary.

For concurrent access, Graflow provides **concurrency-safe primitives** ([Concurrency Safety](https://graflow.ai/docs/getting-started/features#concurrency-safety)):

| Primitive | Use Case | MemoryChannel | RedisChannel |
|---|---|---|---|
| `atomic_add(key, amount)` | Counters, metrics | per-key RLock | INCRBYFLOAT |
| `append(key, value)` | Log collection, result aggregation (FIFO) | GIL + setdefault | RPUSH |
| `prepend(key, value)` | Priority queues, LIFO stacks | GIL + setdefault | LPUSH |
| `lock(key)` | Conditional updates, multi-key consistency | per-key RLock | Distributed lock |

With the RedisChannel backend, Redis's single-threaded execution model serializes all read/write requests, so no application-level locking is required.

### Channel Basics: `set` / `get`

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("channel_demo") as ctx:

    @task(inject_context=True)
    def producer(context: TaskExecutionContext):
        channel = context.get_channel()
        channel.set("config", {"batch_size": 100})
        channel.set("counter", 1)
        print(f"  Producer: config={channel.get('config')}, counter={channel.get('counter')}")

    @task(inject_context=True)
    def consumer(context: TaskExecutionContext):
        channel = context.get_channel()
        config = channel.get("config")  # {"batch_size": 100}
        counter = channel.get("counter")  # 1
        channel.set("counter", counter + 1)  # Explicit update
        print(f"  Consumer: config={config}, counter={channel.get('counter')}")

    producer >> consumer
    ctx.execute("producer")

### Automatic Keyword Argument Resolution

No need for `inject_context` at all! Task argument names are automatically matched to channel keys.

**How it works:** Graflow inspects the task function's signature via `inspect.signature()`. If a channel key matches an argument name, the value is injected automatically. Arguments with default values fall back to their defaults when the key is absent; arguments without defaults raise an error if the key is missing.

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("auto_resolve") as ctx:

    @task(inject_context=True)
    def setup(context: TaskExecutionContext):
        channel = context.get_channel()
        channel.set("user_name", "Alice")
        channel.set("user_city", "Tokyo")

    @task
    def greet(user_name: str, user_city: str = "Unknown"):
        # No inject_context needed — argument names match channel keys
        print(f"  Hello, {user_name} from {user_city}!")

    setup >> greet
    ctx.execute("setup")

### TypedChannel — Type-Safe Data Sharing

Define a schema with `TypedDict` and retrieve a typed channel via `get_typed_channel()` to enable IDE **autocompletion and type checking**. This catches key typos and type mismatches at coding time, making data handling safe even in large-scale workflows.

In [ ]:
from graflow import TaskExecutionContext, task, workflow
from typing import TypedDict

class UserProfile(TypedDict):
    user_id: str
    name: str
    email: str

with workflow("typed_channel_demo") as ctx:

    @task(inject_context=True)
    def collect_user(context: TaskExecutionContext):
        profile_ch = context.get_typed_channel(UserProfile)
        profile_ch.set(
            "current_user",
            {
                "user_id": "u_123",
                "name": "Alice",
                "email": "alice@example.com",
            },
        )
        print("  User profile saved")

    @task(inject_context=True)
    def display_user(context: TaskExecutionContext):
        profile_ch = context.get_typed_channel(UserProfile)
        user = profile_ch.get("current_user")
        print(f"  User: {user['name']} ({user['email']})")

    collect_user >> display_user
    ctx.execute("collect_user")

---

## 4. Branching and Loops — Dynamic Control Without Upfront Definitions

Graflow lets you control branching and loops **dynamically at runtime**. There is no need to predefine every possible path.

- `next_task()`: Add a new task at runtime (dynamic branching)
- `next_iteration()`: Re-execute the current task (retry, convergence loops)

### `next_iteration()` — Retry (Re-execute the Current Task)

In [ ]:
from graflow import TaskExecutionContext, task, workflow
import random

with workflow("retry_demo") as ctx:

    @task(inject_context=True)
    def process_data(context: TaskExecutionContext):
        channel = context.get_channel()
        attempt = channel.get("attempt", 0) + 1
        channel.set("attempt", attempt)

        # Simulate a random score (higher attempts improve success probability)
        score = random.random() * (0.5 + attempt * 0.15)
        score = min(score, 1.0)
        print(f"  Attempt {attempt}: score = {score:.2f}")

        if score < 0.8 and attempt < 5:
            # Retry: re-execute this task
            print("  → Score too low, retrying")
            context.next_iteration()
        else:
            channel.set("final_score", score)
            print(f"  → Success! Final score: {score:.2f}")

    @task
    def finalize(final_score: float):
        print(f"  Done: final score = {final_score:.2f}")

    process_data >> finalize

    random.seed(42)
    ctx.execute("process_data")

### `next_task()` — Dynamic Branching (Generate New Tasks at Runtime)

`next_task()` **creates new tasks and adds them to the graph** at runtime. This is useful when task creation depends on runtime information such as file counts or API responses.

In [ ]:
from graflow import TaskExecutionContext, task, workflow
from graflow.core.task import TaskWrapper

with workflow("dynamic_fanout") as ctx:

    @task(inject_context=True)
    def discover_files(context: TaskExecutionContext):
        """File count is determined at runtime; tasks are generated accordingly"""
        files = ["report.csv", "users.json", "logs.txt"]
        print(f"  Discovered {len(files)} files")

        # Dynamically generate tasks based on file count (Fan-out)
        for file in files:
            context.next_task(
                TaskWrapper(
                    f"process_{file}",
                    lambda f=file: print(f"  Processing: {f}"),
                )
            )

    discover_files  # Only the entry-point task is defined; successors are generated at runtime

    ctx.execute("discover_files")

### Dynamic Control Method Reference

| Method | Behavior | Use Case |
|---|---|---|
| `next_task(task)` | Add a new task | Dynamic branching, Fan-out |
| `next_task(task, goto=True)` | Jump to an existing task (skip successors) | Early exit, error handling |
| `next_iteration()` | Re-execute the current task | Retry, convergence loops |
| `terminate_workflow()` | Graceful termination | Early completion |
| `cancel_workflow(reason)` | Abort with error | Error-triggered cancellation |

### `max_cycles` — Controlling Iteration Count

In addition to manual loops via `next_iteration()`, you can declaratively set an iteration limit with `@task(max_cycles=N)`. Use `ctx.can_iterate()` to check remaining cycles and `ctx.cycle_count` to get the current cycle number (1-based).

In [ ]:
from graflow import task
from graflow.core.context import ExecutionContext, TaskExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph

graph = TaskGraph()

@task(inject_context=True, max_cycles=20)
def optimizer(ctx: TaskExecutionContext, data=None):
    loss = (data or {}).get("loss", 1.0) * 0.5
    print(f"  cycle {ctx.cycle_count}: loss={loss:.4f}")
    if loss < 0.05:
        print(f"  Converged at cycle {ctx.cycle_count}")
        return
    if ctx.can_iterate():
        ctx.next_iteration({"loss": loss})

graph.add_node(optimizer, "optimizer")
WorkflowEngine().execute(
    ExecutionContext.create(graph, start_node="optimizer")
)

### `RetryPolicy` — Automatic Retry on Failure

Specify a `RetryPolicy` directly on the `@task` decorator, with options like `jitter` (randomization) and `max_interval` (backoff cap). You can also set `default_max_retries` to configure a workflow-wide default.

In [ ]:
from graflow import RetryPolicy, task
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.graph import TaskGraph
from graflow.exceptions import GraflowRuntimeError

graph = TaskGraph()
attempts = 0

@task(
    retry_policy=RetryPolicy(
        max_retries=3,
        initial_interval=0.1,  # short for demo
        backoff_factor=2.0,    # 0.1s → 0.2s → 0.4s
    ),
)
def flaky_service():
    global attempts
    attempts += 1
    print(f"  attempt {attempts}")
    if attempts < 3:
        raise ConnectionError(f"timeout (attempt {attempts})")
    return "success"

graph.add_node(flaky_service, "flaky_service")
context = ExecutionContext.create(graph, start_node="flaky_service")
WorkflowEngine().execute(context)
print(f"  Recovered after {attempts} attempts, result: {context.get_result('flaky_service')}")

---

## 5. Parallel Group Error Policies

Graflow provides **flexible error handling at the parallel group level**.

| Policy | Behavior | Example Use Case |
|---|---|---|
| **Strict** (default) | All tasks must succeed | Financial transactions, data validation |
| **Best-effort** | Continue on partial success | Notifications, optional processing |
| **At-least-N** | Specify minimum success count | Multi-region deployments |
| **Critical** | Only evaluate critical tasks | Required + optional task mix |

In [ ]:
from graflow import task, workflow
from graflow.coordination.executor import CoordinationBackend
from graflow.core.handlers.group_policy import (
    BestEffortGroupPolicy,
)

with workflow("error_policy_demo") as ctx:

    @task
    def send_email():
        print("  Email sent successfully")

    @task
    def send_sms():
        raise RuntimeError("SMS delivery failed!")  # Intentional failure

    @task
    def send_slack():
        print("  Slack message sent successfully")

    @task
    def done():
        print("  Notification complete (continued despite partial failure)")

    # BestEffortPolicy: continue even if some tasks fail
    (send_email | send_sms | send_slack).with_execution(
        backend=CoordinationBackend.THREADING,
        policy=BestEffortGroupPolicy(),
    ) >> done

    ctx.execute("send_email")

---

## 6. Hands-on Exercise — Data Analysis Pipeline

A practical pipeline that combines everything covered so far:

- `@task` decorator with `>>` / `|` operators
- Channel-based inter-task data sharing
- Automatic keyword argument resolution (`generate_report` arguments are auto-resolved from the channel)
- Diamond pattern (Fan-out → Fan-in)

In [ ]:
from graflow import TaskExecutionContext, task, workflow

with workflow("data_analysis") as ctx:

    @task(inject_context=True)
    def fetch_data(context: TaskExecutionContext):
        """Fetch data and store it in the channel"""
        data = {
            "sales": [100, 200, 150, 300, 250],
            "costs": [50, 80, 60, 120, 100],
        }
        channel = context.get_channel()
        channel.set("raw_data", data)
        print(f"📥 Data fetched: {len(data['sales'])} records")

    @task(inject_context=True)
    def analyze_sales(context: TaskExecutionContext):
        """Sales analysis (parallel task 1)"""
        channel = context.get_channel()
        sales = channel.get("raw_data")["sales"]
        total = sum(sales)
        channel.set("sales_total", total)
        print(f"📊 Sales analysis: total={total}, avg={total / len(sales)}")

    @task(inject_context=True)
    def analyze_costs(context: TaskExecutionContext):
        """Cost analysis (parallel task 2)"""
        channel = context.get_channel()
        costs = channel.get("raw_data")["costs"]
        total = sum(costs)
        channel.set("cost_total", total)
        print(f"💰 Cost analysis: total={total}, avg={total / len(costs)}")

    @task
    def generate_report(sales_total: int, cost_total: int):
        """Combine analysis results into a report (auto keyword argument resolution)"""
        profit = sales_total - cost_total
        margin = (profit / sales_total) * 100
        print("\n📝 === Analysis Report ===")
        print(f"   Total sales: {sales_total}")
        print(f"   Total costs: {cost_total}")
        print(f"   Profit: {profit} (margin: {margin:.1f}%)")

    # Workflow definition — readable structure in one line
    fetch_data >> (analyze_sales | analyze_costs) >> generate_report

    ctx.execute("fetch_data")

---

## 7. Checkpoint/Resume — Suspend and Resume Workflows

For long-running workflows, the ability to save progress and resume later is essential. Graflow lets **you choose exactly where to save**, keeping checkpoints efficient.

**Key points:**
- Create a checkpoint with `task_ctx.checkpoint(path=...)`
- Restore with `CheckpointManager.resume_from_checkpoint(path)`
- Task results (channel data) are included in the save and restore

In [ ]:
import os
import tempfile

from graflow import TaskExecutionContext, task, workflow
from graflow.core.checkpoint import CheckpointManager
from graflow.core.engine import WorkflowEngine

# Save checkpoints to a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    checkpoint_path = os.path.join(tmpdir, "demo_checkpoint")

    # === Part 1: Run workflow + create checkpoint ===
    with workflow("checkpoint_demo") as wf:

        @task
        def step_1() -> str:
            print("Step 1: Preparing data")
            return "data_prepared"

        @task(inject_context=True)
        def step_2(task_ctx: TaskExecutionContext) -> str:
            print("Step 2: Processing")
            # Create a checkpoint after this task completes
            task_ctx.checkpoint(path=checkpoint_path, metadata={"stage": "processing_done"})
            return "data_processed"

        @task
        def step_3() -> str:
            print("Step 3: Finalizing")
            return "completed"

        step_1 >> step_2 >> step_3

        # Execute (ret_context=True to also retrieve the context)
        _result, context = wf.execute("step_1", ret_context=True)

    print(f"\nCheckpoint saved to: {context.last_checkpoint_path}")
    print(f"Steps executed: {context.steps}")

    # === Part 2: Restore from checkpoint ===
    print("\n--- Restoring from checkpoint ---")
    restored_ctx, metadata = CheckpointManager.resume_from_checkpoint(
        context.last_checkpoint_path
    )

    print(f"Restored tasks: {list(restored_ctx.completed_tasks)}")
    print(f"step_1 result: {restored_ctx.get_result('step_1')}")
    print(f"step_2 result: {restored_ctx.get_result('step_2')}")

    # Resume execution from the restored context (starting at step_3)
    print("\n--- Resuming execution ---")
    engine = WorkflowEngine()
    final_result = engine.execute(restored_ctx)

    print(f"\nstep_3 result: {restored_ctx.get_result('step_3')}")
    print(f"Final result: {final_result}")

---

## 8. Custom Task Handlers — Swappable Execution Strategies

Graflow controls **how** tasks execute through handlers. The default is in-process execution (`direct`), but you can create custom handlers and assign them with `@task(handler="...")` to plug in logging, timing, remote execution, and more.

**How to create a handler:**
1. Subclass `TaskHandler` and implement `execute_task()`
2. Register the handler with `WorkflowEngine`
3. Use it via `@task(handler="name")`

> **Note:** Graflow includes a built-in `DockerTaskHandler` for running tasks in isolated Docker containers.
> Since Docker is unavailable on Colab, this example demonstrates a custom handler instead.
> See [Docker Task Handler](https://graflow.ai/docs/tutorial/advanced/task-handlers#docker-task-handler) for details.

In [ ]:
import time

from graflow import task, workflow
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.core.handler import TaskHandler
from graflow.core.task import Executable

# --- Custom handler: measures task execution time ---
class TimingHandler(TaskHandler):
    """Custom handler that measures task execution time"""

    def execute_task(self, task: Executable, context: ExecutionContext):
        task_id = task.task_id
        start = time.perf_counter()
        try:
            result = task.run()
            elapsed = time.perf_counter() - start
            print(f"  [TimingHandler] {task_id} completed in {elapsed:.3f}s")
            context.set_result(task_id, result)
            return result
        except Exception as e:
            context.set_result(task_id, e)
            raise

# --- Workflow definition ---
with workflow("handler_demo") as ctx:

    @task(handler="direct")  # Default handler
    def fetch_data() -> dict:
        print("fetch_data: Fetching data...")
        return {"values": [1, 2, 3, 4, 5]}

    @task(handler="timing")  # Run with custom handler
    def process_data(fetch_data: dict) -> float:
        print("process_data: Computing...")
        time.sleep(0.1)  # Simulate heavy computation
        return sum(fetch_data["values"]) / len(fetch_data["values"])

    @task(handler="timing")  # Run with custom handler
    def format_result(process_data: float) -> str:
        print("format_result: Formatting...")
        return f"Average: {process_data:.1f}"

    fetch_data >> process_data >> format_result

    # Register custom handler and execute
    engine = WorkflowEngine()
    engine.register_handler("timing", TimingHandler())

    exec_context = ExecutionContext.create(ctx.graph, "fetch_data")
    result = engine.execute(exec_context)

print(f"\nResult: {result}")

---

## 9. LLM Integration — `inject_llm_client`

Graflow provides a unified API for 100+ LLM providers — including Ollama, OpenAI, Claude, and Gemini — through [LiteLLM](https://docs.litellm.ai/).
This tutorial uses **Ollama** to run a local LLM.

#### Recommended Models for Colab

Google Colab's free tier offers **CPU (~12.7 GB RAM)** or a **T4 GPU (15 GB VRAM)**.
Below are representative models available through Ollama:

| Model | Parameters | Size | Context Length | Quantization | Notes |
|--------|------------|------|--------------|--------|------|
| [gemma4:e4b](https://ollama.com/library/gemma4:e4b) | 4.5B (effective) | 9.6 GB | 128K | Q4_K_M | Text, image & audio support. **Used in this tutorial** |
| [llama3.1:8b](https://ollama.com/library/llama3.1:8b) | 8.0B | 4.9 GB | 128K | Q4_K_M | Meta's lightweight, high-performance text model |
| [gpt-oss:20b](https://ollama.com/library/gpt-oss) | 20B (MoE) | 14 GB | 128K | MXFP4 | Requires T4 GPU. Exceeds CPU-only RAM |

> **💡 Tip:** Switching the Colab runtime to **T4 GPU** significantly speeds up inference (Runtime → Change runtime type → T4 GPU). With 15 GB VRAM, all models listed above will fit.

> **This tutorial uses `gemma4:e4b`**, which runs on both CPU and T4 GPU.
> To try a different model, update the model download cell below and the `LLMClient(model=...)` model name.

### Approach 1: `inject_llm_client` (LiteLLM Integration)

Setting `@task(inject_llm_client=True)` automatically injects an `LLMClient` as the task's first argument.
LLMClient wraps [LiteLLM](https://docs.litellm.ai/), providing a unified API for 100+ LLM providers including OpenAI, Claude, and Gemini.

**Key points:**
- `LLMClient` is a **singleton** within a workflow (shared across all tasks)
- Set a default model while retaining the ability to specify a different model per call
- This tutorial uses **Ollama** (local LLM), so no API key is needed
- When `inject_llm_client=True` is set and no `LLMClient` is registered, a default client is auto-created using the `GRAFLOW_LLM_MODEL` environment variable

In [ ]:
!pip install -q litellm

# --- Ollama setup (for Google Colab) ---
# Install and start Ollama locally on Colab.
# No API key required.

import os
import time

# Install zstd (required by the Ollama installer)
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null 2>&1

# Install CUDA drivers (needed for Ollama to detect T4 GPU)
!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get install -y -qq cuda-drivers > /dev/null 2>&1

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama server in the background
!nohup ollama serve &
time.sleep(3)  # Wait for server to start

# Configure LiteLLM to use Ollama
os.environ["OLLAMA_API_BASE"] = "http://localhost:11434"

print("Ollama server ready!")

In [ ]:
# Download the model (takes a while on first run)
!ollama pull gemma4:e4b
# !ollama pull llama3.1:8b

In [ ]:
from graflow import task, workflow
from graflow.core.context import ExecutionContext, TaskExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.llm.client import LLMClient

with workflow("llm_client_demo") as ctx:
    # Create an LLMClient (shared across all tasks)
    llm_client = LLMClient(model="ollama/gemma4:e4b", enable_tracing=False)

    @task(inject_context=True, inject_llm_client=True)
    def summarize(ctx: TaskExecutionContext, llm_client: LLMClient):
        """LLMClient is auto-injected; the result is stored as 'summary' in the channel"""
        result = llm_client.completion_text(
            messages=[{"role": "user", "content": "Explain Python in one sentence"}],
        )
        ctx.get_channel().set("summary", result)

    @task(inject_llm_client=True)
    def translate(llm_client: LLMClient, summary: str) -> str:
        """Auto-resolves 'summary' from the channel + LLMClient injection"""
        return llm_client.completion_text(
            messages=[{"role": "user", "content": f"Translate to Japanese: {summary}"}],
        )

    # summarize stores "summary" in the channel,
    # which is auto-resolved as translate's summary argument
    summarize >> translate

    # Pass llm_client to ExecutionContext and run
    exec_context = ExecutionContext.create(ctx.graph, "summarize", llm_client=llm_client)
    result = WorkflowEngine().execute(exec_context)

print(f"Translation result: {result}")

#### Auto-Created LLMClient

Even without passing `llm_client` to `ExecutionContext.create()`, a **default LLMClient is auto-created** when a task with `inject_llm_client=True` runs.
The default model is configurable via the `GRAFLOW_LLM_MODEL` environment variable. Here we set it to Ollama's `gemma4:e4b`.

In [ ]:
from graflow import task, workflow
from graflow.llm.client import LLMClient

import os
os.environ["GRAFLOW_LLM_MODEL"] = "ollama/gemma4:e4b"

with workflow("llm_auto_demo") as ctx:
    # No explicit llm_client — auto-created using the GRAFLOW_LLM_MODEL env var

    @task(inject_llm_client=True)
    def ask(llm_client: LLMClient) -> str:
        print(f"Auto-created model: {llm_client.model}")
        return llm_client.completion_text(
            messages=[{"role": "user", "content": "What is 1+1? Answer in one word."}],
        )

    result = ctx.execute("ask")

print(f"Answer: {result}")

#### Switching Models Per Call

You can use different models within the same task on a per-call basis.

In [ ]:
from graflow import task, workflow
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.llm.client import LLMClient

with workflow("llm_model_switch") as ctx:
    llm_client = LLMClient(model="ollama/gemma4:e4b", enable_tracing=False)

    @task(inject_llm_client=True)
    def multi_model(llm_client: LLMClient) -> dict:
        # Use the default model (ollama/gemma4:e4b)
        quick = llm_client.completion_text(
            messages=[{"role": "user", "content": "Say hello in Japanese"}],
        )
        # Specify a different model for this call
        detailed = llm_client.completion_text(
            messages=[{"role": "user", "content": "Explain quantum computing in one sentence"}],
            model="ollama/gemma4:e4b",  # Explicit model override (same model here for demo)
        )
        return {"quick": quick, "detailed": detailed}

    # Pass llm_client to ExecutionContext and run
    exec_context = ExecutionContext.create(ctx.graph, "multi_model", llm_client=llm_client)
    result = WorkflowEngine().execute(exec_context)

print(f"Result 1: {result['quick']}")
print(f"Result 2: {result['detailed']}")

### Approach 2: `inject_llm_agent` (Google ADK Integration)

Setting `@task(inject_llm_agent="agent_name")` auto-injects an `LLMAgent` registered in the `ExecutionContext`.
Using `AdkLLMAgent`, which wraps Google ADK's `LlmAgent`, you can embed agents with **tool calling (ReAct pattern)** into your workflows.

In [ ]:
from google.adk.agents import LlmAgent
from graflow import task, workflow
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.llm.agents import AdkLLMAgent, LLMAgent


# ADK tool definition (a plain function becomes a tool)
def get_weather(city: str) -> dict:
    """Get the weather for a given city."""
    # Dummy data for demo purposes
    weather_data = {
        "Tokyo": {"temp": 22, "condition": "Sunny"},
        "Osaka": {"temp": 25, "condition": "Cloudy"},
        "Sapporo": {"temp": 15, "condition": "Rainy"},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
    return {"city": city, "temp_celsius": data["temp"], "condition": data["condition"]}


with workflow("adk_agent_demo") as ctx:
    # Create an ADK agent with tools
    adk_agent = LlmAgent(
        name="weather_agent",
        model="ollama/gemma4:e4b",  # model="gemini-2.5-flash",
        instruction="You are a weather assistant. Use the get_weather tool to answer questions.",
        tools=[get_weather],
    )

    @task(inject_llm_agent="weather_agent")
    def ask_weather(llm_agent: LLMAgent) -> str:
        """The agent calls tools to fetch and summarize weather information"""
        result = llm_agent.run("What's the weather in Tokyo and Sapporo?")
        return result["output"]

    # Create ExecutionContext and register the agent
    exec_context = ExecutionContext.create(ctx.graph, "ask_weather")
    agent = AdkLLMAgent(adk_agent, app_name=exec_context.session_id, enable_tracing=False)
    exec_context.register_llm_agent("weather_agent", agent)

    result = WorkflowEngine().execute(exec_context)

print(f"Weather report:\n{result}")

### Approach 3: `inject_llm_agent` (Pydantic AI Integration)

You can also use [Pydantic AI](https://ai.pydantic.dev/) agents via `PydanticLLMAgent`.
Pydantic AI provides **type-safe structured outputs** using Pydantic models and supports tool calling across multiple LLM providers through LiteLLM.

Key differences from ADK:
- **Structured output** via `output_type` — the agent's response is automatically validated as a Pydantic `BaseModel`
- **LiteLLM routing** — unified model access across OpenAI, Anthropic, Google, and more
- **Tool registration** via `@agent.tool` decorator with `RunContext` for dependency injection

In [ ]:
from pydantic import BaseModel
from pydantic_ai import RunContext
from graflow import task, workflow
from graflow.core.context import ExecutionContext
from graflow.core.engine import WorkflowEngine
from graflow.llm.agents import LLMAgent, PydanticLLMAgent, create_pydantic_ai_agent


# Structured output schema — the agent's response is validated automatically
class WeatherReport(BaseModel):
    summary: str
    cities: list[dict]


with workflow("pydantic_agent_demo") as ctx:
    # Create a Pydantic AI agent with LiteLLM backend and structured output
    pydantic_agent = create_pydantic_ai_agent(
        model="ollama/gemma4:e4b",  # or "openai/gpt-4o", etc.
        output_type=WeatherReport,  # Type-safe structured output
        system_prompt="You are a weather assistant. Use the get_weather tool to answer questions.",
    )

    # Register tools via @agent.tool decorator (supports RunContext for dependency injection)
    @pydantic_agent.tool
    def get_weather(ctx: RunContext[None], city: str) -> dict:
        """Get the weather for a given city."""
        weather_data = {
            "Tokyo": {"temp": 22, "condition": "Sunny"},
            "Osaka": {"temp": 25, "condition": "Cloudy"},
            "Sapporo": {"temp": 15, "condition": "Rainy"},
        }
        data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
        return {"city": city, "temp_celsius": data["temp"], "condition": data["condition"]}

    @task(inject_llm_agent="weather_agent")
    def ask_weather(llm_agent: LLMAgent) -> str:
        """The agent calls tools and returns a validated WeatherReport"""
        result = llm_agent.run("What's the weather in Tokyo and Sapporo?")
        report: WeatherReport = result["output"]  # Type-safe Pydantic model
        return report.summary

    # Create ExecutionContext and register the Pydantic AI agent
    exec_context = ExecutionContext.create(ctx.graph, "ask_weather")
    agent = PydanticLLMAgent(pydantic_agent, name="weather_agent", enable_tracing=False)
    exec_context.register_llm_agent("weather_agent", agent)

    result = WorkflowEngine().execute(exec_context)

print(f"Weather report:\n{result}")

---

## Summary

This tutorial walked you through Graflow's core features.

### Feature Overview

| Feature | Description |
|---|---|
| **Graph Definition** | `>>` / `\|` operators (readable one-liner syntax) + low-level API |
| **Data Sharing** | Channel (key-value) + automatic keyword argument resolution |
| **Branching** | `next_task()` / `next_iteration()` (dynamic at runtime) |
| **Checkpointing** | User-controlled (save and restore at any point) |
| **Parallel Error Handling** | 4 built-in policies + custom |
| **Task Handlers** | direct / docker / custom |
| **LLM Integration** | LiteLLM + Google ADK agents + Pydantic AI agents |
| **Tracing** | Langfuse (OSS) + OpenTelemetry |
| **Distributed Execution** | Horizontal scaling via Redis-based workers |
| **Design Philosophy** | Define-by-Run (build the graph as you execute) |

For more details, see the [Graflow documentation](https://graflow.ai/docs).
